# Week 3 — Revision Topics & Concepts

This notebook is a compact revision sheet for Week 3: a list of the exact topics interviewers love to ask about, each explained head-on and backed by short runnable cells. It opens with container fundamentals — **why sets cannot be indexed** (they are unordered, membership lives in hashes, not positions) and **why dictionary keys cannot be mutable** (a mutable key could change its own hash and corrupt the lookup table). Next you revise `enumerate()`, which pairs every item with its index, and the constructor/destructor pair `__init__` / `__del__`, which hook object creation and destruction. From there the sheet walks through the three introspection helpers `dir()`, `isinstance()`, and `issubclass()`, and draws the classic contrast between **class methods** and **static methods** — that single difference of who-or-what gets passed as the first argument.

The second half digs into the deeper object-model questions. The **diamond problem** shows what happens when multiple inheritance creates an ambiguous method, and how Python's C3 **Method Resolution Order** resolves it predictably. The underscore briefing explains the five naming patterns and the name-mangling behind `__var`. Then come the two magic-method pairs that trip everyone up: **`repr` vs `str`** (developer representation versus human-friendly description) and the combination of **`__eq__` and `__hash__`** — which is also the answer to the famous question *how can mutable objects be stored in sets at all?*

Work through the topics in order; each has a theory cell followed by code that demonstrates the claim. Run the cells, read the outputs, and use the blank cells in-between for your own experiments. By the end you will have a compact, accurate picture of hashing, name mangling, method binding, and introspection — the exact ground where interviewers probe.

### Topics covered
- Set indexing and dict-key mutability
- `enumerate`, destructors, `dir`/`isinstance`/`issubclass`
- Class method vs static method, the diamond problem
- Underscores, `repr` vs `str`, mutable objects in sets

# 1. Set Indexing & Dict Keys

## 1.1 Why sets cannot be indexed

A **set** is an unordered collection, and "unordered" is not a small detail — it is the definition. There is no position "0" in a set because the elements are organised by their **hash values** for lightning-fast membership tests, not by any sequence. `s[0]` has no meaning by construction; to read from a set you iterate it (`for x in s`) or test membership (`x in s`). Even the printed order of the demo set `{21, 34, 11, 56, 39}` looks sorted by coincidence of hashing — never rely on it.

A **dictionary key** obeys the same hashing law: the key must be *hashable*, and Python's rule of thumb is that hashable means immutable. A tuple such as `(1, 2, 3)` is immutable, so it is a legal dict key — the demo succeeds. A list is mutable: if we used `[1, 2, 3]` as a key and then appended to it, the key's hash would change *after* it was already placed in the table, and the dict would lose track of the entry. Python therefore refuses the whole idea at insertion time with a `TypeError: unhashable type: 'list'`, which the code cell below reproduces. Remembering that sets and dicts are both hash tables explains both questions in one breath.

In [97]:
s = {21,34,11,56,39}
s

{11, 21, 34, 39, 56}

In [98]:
d = {(1,2,3):'nitish'}
d

{(1, 2, 3): 'nitish'}

In [99]:
d = {[1,2,3]:'nitish'}
d

TypeError: ignored

# 2. enumerate()

## 2.1 Adding a counter to any iterable

`enumerate(iterable)` takes any iterable and returns pairs of `(index, value)` — it attaches a counter as you loop. The pairs are produced lazily (an *enumerate object*), so you usually consume them in a `for` loop or convert with `list()`. The first demo sorts a list of `(name, score)` tuples by score using a `key` function that picks element `[1]`, with `reverse=True` for descending order — giving the ranking `nitish, ankita, ankit`.

The second demo shows the tricky part interviewers probe: `sorted(list(enumerate(L)), reverse=True)` sorts the enumerate pairs as *tuples*, so Python compares the **index first**. For `L = [15, 21, 13, 13]` the pairs are `(0, 15), (1, 21), (2, 13), (3, 13)`; a reverse sort ordered primarily by index produces `(3, 13), (2, 13), (1, 21), (0, 15)` — the values are *descending by index*, not by score. When you actually want ranking by value, reach for the `key=` approach of the first demo instead.

In [112]:
# enumerate
# The enumerate() method adds a counter to an iterable and returns it (the enumerate object).
L = [('nitish',45),('ankit',31),('ankita',40)]

sorted(L,key=lambda x:x[1],reverse=True)

[('nitish', 45), ('ankita', 40), ('ankit', 31)]

In [107]:
L = [15,21,13,13]
sorted(list(enumerate(L)),reverse=True)

[(3, 13), (2, 13), (1, 21), (0, 15)]

# 3. Constructors and Destructors

## 3.1 Hooks at the ends of an object's life

A Python object's life is bracketed by two dunder methods. The **constructor** `__init__` is where we set up the instance — it runs automatically the moment the object is created and receives the freshly allocated `self`. The **destructor** `__del__` is its complement: it is called when the object is about to be destroyed, usually by the garbage collector once no references remain.

The demo illustrates the reference-counting subtlety. `obj = Example()` creates one object with one reference; `a = obj` adds a second reference; `del obj` removes one but the object survives because `a` still points to it — nothing happens yet. Only after `del a` releases the last reference does the destructor fire and print `destructor called`. That is why you never know *exactly when* `__del__` runs: it depends on references, not on your code. In modern Python, resources are usually released with context managers (`with`) rather than `__del__`, but knowing the lifecycle is still standard interview material.

In [116]:
# destructor
class Example:

  def __init__(self):
    print('constructor called')

  # destructor
  def __del__(self):
    print('destructor called')

obj = Example()
a = obj
del obj
del a

constructor called
destructor called


# 4. dir, isinstance, issubclass

## 4.1 Three introspection helpers

Python makes its own objects inspectable at runtime, and three built-ins cover the basics:

- `dir(obj)` — lists the *members*: attributes, methods, and dunders belonging to an object. In the demo, `dir(t)` on the `Test` instance shows name-mangled entries such as `_Test__baz` alongside the public `foo` and `_bar`, proving that mangling is visible through introspection.
- `isinstance(obj, ClassName)` — asks **"is this object an instance of this class (or its subclasses)?"** It returns `True` for the freshly created `Example` object; it works for built-ins too (`isinstance(5, int)`).
- `issubclass(B, A)` — asks **"is this class a subclass of that class?"**, comparing two classes, not an object and a class.

The clean mental split: `dir` shows *what an object has*, `isinstance` checks *what an object is*, and `issubclass` checks *how two classes relate*. They are the first tools you reach for when you are exploring an unfamiliar codebase or writing generic code that must handle several types.

In [ ]:
# dir

class Test:
    def __init__(self):
        self.foo = 11
        self._bar = 23
        self.__baz = 23

    def greet(self):
      print('hello')
    
t = Test()
print(dir(t)) # This gives us a list with the objectâ€™s attributes

['_Test__baz', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_bar', 'foo', 'greet']


In [ ]:
# isinstance

class Example:

  def __init__(self):
    print('hello')

obj = Example()

isinstance(obj,Example)

hello


True

In [118]:
# issubclass
class A:
  def __init__(self):
    pass

class B(A):
  pass

issubclass(B,A)

True

# 5. Class Methods vs Static Methods

## 5.1 Who gets passed as the first argument

Both `@classmethod` and `@staticmethod` belong to the class rather than to a particular instance, and both can be called on the class itself without creating an object. The difference is the first parameter:

- A **class method** receives `cls` — the class itself. It therefore *can* read and modify class state (class variables shared by every instance), which is why it is used for factory methods and alternative constructors (`Person.from_dict(...)`).
- A **static method** receives nothing implicit. It behaves like a plain function that merely lives inside the class's namespace — it cannot see `self` or `cls`, and it can neither read nor modify class state. It is the right tool for utility functions that logically belong to the class.

The demo class `A` defines all three kinds. Calling them through an instance works for each (`a.normal_m()`, `a.class_m()`, `a.static_m()`), and calling them through the class works for the class and static versions (`A.static_m()`, `A.class_m()`) while `A.normal_m()` fails — a plain instance method needs a real instance for `self`, though you can always pass one explicitly: `A.normal_m(a)`.

### staticmethod
A static method does not receive an implicit first argument. A static method is also a method that is bound to the class and not the object of the class. This method canâ€™t access or modify the class state. It is present in a class because it makes sense for the method to be present in class.

In [92]:
class A:

  def normal_m(self):
    print('normal method')

  @staticmethod
  def static_m():
    print('static method')
  
  @classmethod
  def class_m(cls):
    print('class method')

In [93]:
a = A()

# normal -> object -> callable
a.normal_m()
# class -> object -> callable
a.class_m()
# static -> object -> not callable
a.static_m()

normal method
class method
static method


In [94]:
# static -> class -> callable
A.static_m()
# class method -> class -> callable 
A.class_m()
# normal -> class -> not callable
A.normal_m()

static method
class method


TypeError: ignored

In [ ]:
# Alternate syntax
A.normal_m(a)

### Class method vs Static Method<br>
The difference between the Class method and the static method is:

- A class method takes cls as the first parameter while a static method needs no specific parameters.
- A class method can access or modify the class state while a static method canâ€™t access or modify it.
- In general, static methods know nothing about the class state. They are utility-type methods that take some parameters and work upon those parameters. On the other hand class methods must have class as a parameter.
- We use @classmethod decorator in python to create a class method and we use @staticmethod decorator to create a static method in python.

### When to use the class or static method?
- We generally use the class method to create factory methods. Factory methods return class objects ( similar to a constructor ) for different use cases.
- We generally use static methods to create utility functions.

# 6. MRO and the Diamond Problem

## 6.1 Resolving multiple inheritance

Under multiple inheritance a method can exist in more than one ancestor, and Python needs a deterministic winner. The decision rule is the **Method Resolution Order**: the child class first, then base classes left to right, following the C3 linearization. `Class4.__mro__` spells out the full chain, from `Class4` up to `object`.

The **diamond problem** is the classic shape: `Class2` and `Class3` both inherit from `Class1`, and `Class4` inherits from both (`Class4(Class3, Class2)`). Which `m()` should `obj.m()` call? Python's left-to-right rule answers without ambiguity: the bases listed first win, so `Class3.m()` executes and the output is `In Class3`. The same deterministic ordering is why Python, unlike some languages, simply does not suffer from the diamond ambiguity — it always knows who is next in line. Interviewers want you to state the rule (left-to-right, child first) and to name the mechanism (C3 linearization, inspectable through `__mro__`).

In [120]:
# The diamond problem
class Class1:
    def m(self):
        print("In Class1")
       
class Class2(Class1):
    def m(self):
        print("In Class2")
 
class Class3(Class1):
    def m(self):
        print("In Class3") 
        
class Class4(Class3, Class2):
    pass  
     
obj = Class4()
obj.m()
# MRO

In Class3


In [ ]:
# Double and single score

# 7. str vs repr

## 7.1 Two ways for an object to describe itself

`str()` and `repr()` are two distinct conversions, and every object gets both for free. `str(obj)` produces the *human-readable* description — what `print(obj)` and f-string interpolation show. `repr(obj)` produces the *developer-facing, unambiguous* description, the one shown when you evaluate a bare expression in the REPL and the one used inside containers and debugging output.

The demos make the difference concrete. For the string `a = 'hello'`, `str(a)` prints `hello` (no quotes, friendly), while `repr(a)` prints `'hello'` (quoted, so you can tell it is a string). With `datetime`, `str(datetime.now())` renders a readable timestamp, whereas `repr(datetime.now())` shows `datetime.datetime(2022, 11, 26, 15, 46, 52, 7475)` — an unambiguous expression that includes the type and all fields. Precisely because `repr` must be unambiguous, custom classes are advised to implement `__repr__` first: `str()` will fall back to it automatically, while the reverse is not true.

In [ ]:
# repr and other magic/dunder methods

a = 'hello'

print(str(a))
print(repr(a))

hello
'hello'


In [123]:
import datetime

a = datetime.datetime.now()
b = str(a)

print(str(a))
print(str(b))

print(repr(a))
print(repr(b))

2022-11-26 15:46:52.007475
2022-11-26 15:46:52.007475
datetime.datetime(2022, 11, 26, 15, 46, 52, 7475)
'2022-11-26 15:46:52.007475'


### In summary

- str is for users -> meant to be more readable
- repr is for developers for debugging - > for being unambigous

## 8.2 Mutable objects in sets, explained

The question "how can mutable objects be stored in sets even though they are mutable?" has a precise answer: sets (and dict keys) require objects to be **hashable**, not immutable. An object is hashable when it implements `__hash__` (and `__eq__`, if defining equality). User-defined classes inherit an *identity-based* hash from `object`, so instances hash like `id(obj)`'s — different for every object, and stable for as long as the object lives.

The first cell below exploits exactly that: `class A` defines `__init__` and `hello` only, an instance `a` is created, and `s = {a}` succeeds — the set stores the object by its identity hash. `dir(a)` shows the inherited dunder hash machinery. The second cell defines `__eq__` inside `A`: the moment you define equality, Python sets `__hash__` to `None` to keep the *hash/eq contract* — objects that compare equal must share a hash. With `__hash__` explicitly redefined back to `1`, the object is hashable again (and `{a}` works), though all instances now collapse to the same bucket. The rule of thumb: define `__eq__`? then define `__hash__` too with your own keys. Real production use rarely needs this, but that is the standard close of the interview line of questioning.

In [127]:
# how objects are stored even though they are mutable
# https://stackoverflow.com/questions/31340756/python-why-can-i-put-mutable-object-in-a-dict-or-set
class A:
  
  def __init__(self):
    print('constructor')

  def hello(self):
    print('hello')

a = A()
a.hello()
s = {a}
print(s)
dir(a)

constructor
hello
{<__main__.A object at 0x7f4f5f3fd510>}


['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'hello']

In [133]:
class A:
  
  def __init__(self):
    print('constructor')

  def __eq__(self):
    pass

  def __hash__(self):
    return 1

  def hello(self):
    print('hello')

a = A()
a.hello()
s = {a}
print(s)

dir(a)



constructor
hello
{<__main__.A object at 0x7f4f5f369290>}


['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'hello']

# 8. Encapsulation & Underscores

## 8.1 Conventions, mangling, and hashing

Query "private" members work on convention. A single underscore — `self._var` — is a request: it signals "internal use", but as the demo shows `a._var` reads perfectly fine, because the interpreter does not enforce anything. A double underscore — `self.__var` — is different: Python rewrites (`mangles`) the name to `_ClassName__var`, which is why `dir()` shows `_Test__baz` and direct access to `__baz` fails. Mangling is Python's gentle collision avoidance for subclass attributes, not real privacy.

The set demos underneath answer a favourite question: *how can mutable objects be stored in sets?* The answer is **hashability**. A set stores objects by their hash, and it requires only that the hash stay stable while the object is in the set — it does *not* require immutability. User-defined objects are hashable by default (their identity hash), so `{a}` works, as the first demo shows. The second demo defines `__eq__` without `__hash__`, which makes Python set `__hash__` to `None` and the object unhashable. Lists fail loudly too: `hash([1, 2, 3])` raises `TypeError` and `{L}` is rejected, while `hash(1)`, `hash('hello')`, and `hash((1, 2, 3))` all succeed. Immutable built-ins are hashable; mutable ones are not — with custom classes, you decide.

In [121]:
class A:

  def __init__(self):
    self._var = 10

a = A()
a._var

10

In [125]:
s = {[1,2]}

TypeError: ignored

In [129]:
L = [1,2,3]
s = {L}

TypeError: ignored

In [137]:
print(L.__hash__)

None


In [138]:
hash(1)

1

In [139]:
hash('hello')

4306082800328210013

In [140]:
hash((1,2,3,))

2528502973977326415

In [141]:
hash([1,2,3])

TypeError: ignored

## Summary

This revision sheet consolidated Week 3's trickiest concepts into digestible, runnable form. You saw why sets cannot be indexed and why dict keys must be immutable-hashable (sets and dicts are hash tables, and a changing hash would break membership). You reviewed `enumerate()`, the `__init__`/`__del__` lifecycle, and the trio `dir` / `isinstance` / `issubclass` for introspection. You distinguished class methods (receive `cls`, can modify class state) from static methods (receive nothing, utilities only), and saw how the C3 MRO resolves the diamond problem deterministically from left to right. The underscore tour separated convention (`_var`), keyword avoidance (`var_`), name mangling (`__var`), and the reserved dunders (`__var__`). Finally, `str` vs `repr` clarified human-friendly versus unambiguous descriptions, and the set demos explained hashability: mutable custom objects can live in sets because sets demand a stable `__hash__`, not immutability — leaving `__eq__`/`__hash__` together as the contract that governs them all.